# 02 · The genome models (`/predict`)

**Forecasting Antibiotic Resistance with AI** · Final Year Project · Hamza Afzal (models), Muhammad Ali Mirza (data and evolution), Suleman Saleh (platform)

`/predict` takes a complete bacterial genome (FASTA) and an antibiotic. Two models are served:

| Model | Features | Lab AUC (unseen genomes) | Served |
|---|---|---|---|
| **Gene model** `G_genes_deploy` | resistance genes and mutations found by NCBI AMRFinderPlus | 0.977 | where AMRFinderPlus is installed (the Docker image) |
| **k-mer model** `G_kmer_deploy` | frequencies of the 256 four-letter DNA words (4-mers) | 0.935 | everywhere else |

This notebook trains both with the project's code, checks them against the published runs, and shows how the website builds the same features from an uploaded genome. Data: 24,926 complete genomes downloaded from BV-BRC by Ali (24,719 after dropping plasmid-only records), labels from the cleaned export. Models are judged on **lab-confirmed rows**: the other labels were predicted by BV-BRC from the genome, so a genome model could score well on them by reproducing that predictor. Run time: about 6 minutes.

In [1]:
import os, sys, json, inspect, time
import numpy as np
import pandas as pd

ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, 'experiments'))
sys.path.insert(0, os.path.join(ROOT, 'experiments', 'genome'))

import run as pipeline
import algorithms
from lib import data_prep, encoders, metrics, splits
import genes, kmers

def show(fn):
    print(inspect.getsource(fn))

## 1. The training procedure

The same steps as the forecaster (notebook 01), written as one function: load and filter the data, attach the genome features, split **by genome**, train LightGBM with early stopping on validation genomes, choose the threshold on validation genomes (at most 10% of resistant isolates missed), and evaluate on the test genomes.

In [2]:
def train_and_evaluate(cfg):
    t0 = time.time()
    dcfg, scfg, fcfg = cfg['data'], cfg['split'], cfg['features']
    df = data_prep.get_clean(source=dcfg.get('source'), verbose=False)
    df = pipeline.compact(df, verbose=False)
    df = pipeline.select_rows(df, dcfg, verbose=False)
    df = pipeline.apply_taxon_level(df, dcfg.get('taxon_level', 'strain'), verbose=False)
    df, genome_features = pipeline.attach_genome_features(df, dcfg, fcfg)      # genes or k-mers

    train_mask, test_mask = splits.make_split(df, strategy=scfg.get('strategy', 'grouped'),
                                              test_size=scfg.get('test_size', 0.2), seed=scfg.get('seed', 42))
    df, rate_features = encoders.add_rate_features(df, train_mask, fcfg.get('target_encoding', 'oof'), None, verbose=False)
    features = list(fcfg.get('base', pipeline.BASE_FEATURES)) + rate_features + genome_features
    cat_features = [c for c in algorithms.CAT_FEATURES if c in features]

    X, y = df[features], df['target'].to_numpy()
    X_train_all, X_test = algorithms.align_categories(X.loc[train_mask], X.loc[test_mask], columns=cat_features)
    y_train_all, y_test = y[train_mask], y[test_mask]
    train_df = df.loc[train_mask].reset_index(drop=True)
    fit_pos, val_pos = splits.inner_folds(train_df, n_splits=6, seed=scfg.get('seed', 42))[0]

    model = algorithms.build_model(cfg['model'])
    model.fit(X_train_all.iloc[fit_pos], y_train_all[fit_pos], X_train_all.iloc[val_pos], y_train_all[val_pos], cat_features)

    tcfg = cfg['threshold']
    threshold = metrics.pick_threshold(y_train_all[val_pos], model.predict_proba(X_train_all.iloc[val_pos]),
                                       strategy=tcfg['strategy'], fixed=tcfg.get('fixed', 0.5), vme_budget=tcfg.get('vme_budget', 0.03))
    score = model.predict_proba(X_test)
    test_rows = df.loc[test_mask]
    by_source = pipeline.metrics_by_label_source(y_test, score, test_rows, threshold, cfg['evaluation'], 'Genome ID')
    print(f"{cfg['id']}: {len(features)} features, {train_mask.sum():,} train rows, "
          f"{test_rows['Genome ID'].nunique():,} test genomes, threshold {threshold:.2f}, {time.time() - t0:.0f} s")
    return {'lab': by_source['lab'], 'all': metrics.evaluate(y_test, score, threshold), 'threshold': threshold,
            'model': model, 'features': features}

def load(name):
    return pipeline.load_config(os.path.join(ROOT, 'experiments', 'configs', name + '.json'))

## 2. Gene features from AMRFinderPlus

Ali ran NCBI AMRFinderPlus (version 4.2.7, database 2026-08-07.1) on every genome. Its hits form the **gene matrix**: one row per genome, one 0/1 column per resistance gene or point mutation. For each (genome, antibiotic) row, `genes.py` adds the raw gene columns (genes found in at least 50 genomes) and drug-aware counts: how many genes and mutations of the drug's class the genome carries, whether a well-known key determinant for that drug is present (for example *gyrA* S83 mutations for ciprofloxacin), and the overall resistance load.

In [3]:
m, info = genes.load_matrix()
print(f'gene matrix: {m.shape[0]:,} genomes x {m.shape[1]:,} genes and mutations '
      f"({(info['type'] == 'point_mutation').sum()} point mutations)")
print('drug class -> AMRFinderPlus classes, e.g.:', {k: sorted(v) for k, v in list(genes.CLASS_TO_AMRFINDER.items())[:4]})
print('key determinants, e.g. ciprofloxacin:', genes.KEY_DETERMINANTS['ciprofloxacin'])
show(genes.drug_aware_features)

gene matrix: 24,926 genomes x 2,733 genes and mutations (1499 point mutations)
drug class -> AMRFinderPlus classes, e.g.: {'beta_lactam': ['BETA-LACTAM'], 'carbapenem': ['BETA-LACTAM'], 'monobactam': ['BETA-LACTAM'], 'fluoroquinolone': ['QUINOLONE']}
key determinants, e.g. ciprofloxacin: ['gyrA_S83', 'gyrA_D87', 'parC_S80', 'qnr']
def drug_aware_features(m, info, pos, antibiotics, drug_classes):
    """Features that depend on the (genome, antibiotic) pair."""
    arr = m.to_numpy(dtype=np.uint8)
    symbols = list(m.columns)
    kinds = info.reindex(symbols)['type'].to_numpy()
    classes = [amr_classes(c) for c in info.reindex(symbols)['class']]
    is_mut = kinds == 'point_mutation'
    n = len(pos)

    gene_n = np.zeros(n, dtype=np.float32)
    mut_n = np.zeros(n, dtype=np.float32)
    for dc in np.unique(drug_classes):
        targets = CLASS_TO_AMRFINDER.get(dc)
        if not targets:
            continue
        cols = np.array([bool(c & targets) for c in classes])
        rows

## 3. Train the gene model (`G_genes_deploy`)

In [4]:
cfg_genes = load('G_genes_deploy')
print(json.dumps({k: cfg_genes[k] for k in ['data', 'features', 'threshold']}, indent=2))
res_genes = train_and_evaluate(cfg_genes)

{
  "data": {
    "genomes": "gene_matrix",
    "min_genome_bp": 500000
  },
  "features": {
    "target_encoding": "none",
    "base": [
      "Antibiotic",
      "drug_class"
    ],
    "genes": {
      "raw": true,
      "min_genomes": 50,
      "drug_aware": true
    }
  },
  "threshold": {
    "strategy": "vme_constrained",
    "vme_budget": 0.1
  }
}


[genome] min_genome_bp 500,000: dropped 7,487,398 rows of 414,823 genomes (plasmid-only records or no assembly)


[genes] 359,712 of 359,712 rows have gene data (24,719 genomes)


[genes] 356 gene columns


[split] grouped: train=287,774 test=71,938 | genomes in both sides: 0


...

[200]	valid_0's auc: 0.975258


[300]	valid_0's auc: 0.977591


[400]	valid_0's auc: 0.97902


[500]	valid_0's auc: 0.979935


G_genes_deploy: 358 features, 287,774 train rows, 4,915 test genomes, threshold 0.58, 197 s


## 4. Train the k-mer model (`G_kmer_deploy`)

A k-mer is a DNA word of length k. With k = 4 there are 256 words; their frequencies in the whole genome are a fixed-size fingerprint, with no alignment or reference needed. Ali's cache holds 6-mer counts; 4-mers are sums of them.

In [5]:
show(kmers.kmer_matrix)
cfg_kmer = load('G_kmer_deploy')
res_kmer = train_and_evaluate(cfg_kmer)

def kmer_matrix(counts6, k=4, canonical=False):
    """(n_genomes, 4**k) k-mer frequencies from 6-mer counts, plus column names.

    Frequencies (counts / total) make genomes of different lengths comparable,
    which is what the shipped model used.

    canonical=True would merge each k-mer with its reverse complement (AAAC
    with GTTT): a contig's strand is arbitrary, so the two are the same
    sequence read from either end. Not implemented yet; see the TODO.
    """
    if not 1 <= k <= K_MAX:
        raise ValueError(f'k must be 1..{K_MAX}')
    c = counts6.reshape((-1,) + (4,) * K_MAX).sum(axis=tuple(range(1 + k, 1 + K_MAX)))
    c = c.reshape(len(counts6), 4 ** k).astype(np.float64)
    names = [f'k{k}_{n}' for n in kmer_names(k)]
    if canonical:
        # TODO(Hamza, B3): merge each k-mer with its reverse complement.
        # 1. complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
        # 2. rc(s) = ''.join(complement[b] for b in reversed(s))
        # 3. keep one colu

[genome] min_genome_bp 500,000: dropped 7,487,398 rows of 414,823 genomes (plasmid-only records or no assembly)


[genome] 359,712 of 359,712 rows have an assembly (24,719 genomes)


[genome] k=4: 258 genome columns


[split] grouped: train=287,774 test=71,938 | genomes in both sides: 0


...

[200]	valid_0's auc: 0.921591


[300]	valid_0's auc: 0.926617


[400]	valid_0's auc: 0.929572


[500]	valid_0's auc: 0.932151


G_kmer_deploy: 259 features, 287,774 train rows, 4,915 test genomes, threshold 0.43, 148 s


## 5. Results on lab-confirmed rows of unseen genomes, and the check against the published runs

In [6]:
pub = lambda run_id: json.load(open(os.path.join(ROOT, 'experiments', 'results', run_id, 'metrics.json')))
rows = []
for name, res in [('G_genes_deploy', res_genes), ('G_kmer_deploy', res_kmer)]:
    p = pub(name)['test_by_label_source']['lab']
    l = res['lab']
    rows.append({'model': name, 'lab rows': l['n'], 'lab AUC': round(l['auc_roc'], 4),
                 '95% CI': f"[{l['auc_roc_ci'][0]:.3f}, {l['auc_roc_ci'][1]:.3f}]",
                 'recall': round(l['sensitivity'], 3), 'specificity': round(l['specificity'], 3),
                 'threshold': round(res['threshold'], 2), 'published lab AUC': round(p['auc_roc'], 4),
                 'same as published': abs(l['auc_roc'] - p['auc_roc']) < 1e-4})
pd.DataFrame(rows)

,model,lab rows,lab AUC,95% CI,recall,specificity,threshold,published lab AUC,same as published
0,G_genes_deploy,40356,0.9772,"[0.975, 0.979]",0.898,0.949,0.58,0.9772,True
1,G_kmer_deploy,40356,0.9345,"[0.931, 0.939]",0.909,0.788,0.43,0.9345,True


For comparison, on the same lab rows: the antibiotic alone scores 0.715, the organism (genus and species) 0.849, and a fixed gene-lookup rule (resistant when a gene of the drug's class is present) 0.711. Learning which genes matter for which drug is what lifts the model to 0.977.

## 6. Is the gene model's lead real? A paired significance test

Both models scored the same 40,356 lab rows. DeLong's test compares their AUCs, McNemar's their calls, and a bootstrap that resamples whole genomes gives a confidence interval that respects genomes having many rows.

In [7]:
import compare
r = compare.compare('B6L_genes_v6', 'B4L_kmer4_lgbm_v6', lab=True, n_boot=200)
d, b, mc = r['delong'], r['bootstrap'], r['mcnemar']
print(f"genes vs k-mers on {r['rows']:,} lab rows ({r['genomes']:,} genomes): AUC {d['auc_a']:.4f} vs {d['auc_b']:.4f}, "
      f"difference {d['diff']:+.4f}, DeLong 95% CI [{d['ci'][0]:+.4f}, {d['ci'][1]:+.4f}], p = {d['p']:.1e}; "
      f"genome bootstrap [{b['ci'][0]:+.4f}, {b['ci'][1]:+.4f}]; McNemar: {mc['only_a_right']:,} rows only genes right, "
      f"{mc['only_b_right']:,} only k-mers right")

genes vs k-mers on 40,356 lab rows (4,481 genomes): AUC 0.9790 vs 0.9345, difference +0.0445, DeLong 95% CI [+0.0426, +0.0464], p = 0.0e+00; genome bootstrap [+0.0411, +0.0478]; McNemar: 4,054 rows only genes right, 1,069 only k-mers right


## 7. How the website builds the features from an uploaded genome

`backend/ml_models/genome_predictor.py` serves both models. For the gene model it (1) identifies the species from the genome's 6-mer profile, (2) runs AMRFinderPlus with that `--organism`, as training did, and (3) builds the gene features. AMRFinderPlus does not run on Windows, so here the hits of a test genome are taken from the gene matrix; the website's feature code then must give **exactly** the training run's probability.

In [8]:
sys.path.insert(0, os.path.join(ROOT, 'backend'))
from ml_models import genome_predictor as gp
from unittest import mock

with mock.patch.object(gp, 'find_amrfinder', return_value='(not needed here)'):
    served = gp.GenomeModelPredictor(os.path.join(ROOT, 'backend', 'trained_models'))
print('served gene model:', served.meta['run_id'], '| features:', len(served.features))

pred = pd.read_csv(os.path.join(ROOT, 'experiments', 'results', 'G_genes_deploy', 'predictions.csv'), dtype={'genome_id': str})
row = pred[(pred.label_source == 'lab') & (pred.antibiotic == 'ciprofloxacin')].iloc[0]
present = m.columns[m.loc[row.genome_id].to_numpy() > 0]
row_info = info.reindex(present)
hits = pd.DataFrame({'symbol': present, 'name': '', 'class': row_info['class'].fillna('').to_numpy(),
                     'subclass': '', 'type': row_info['type'].to_numpy()})
X, _ = served.features_frame('>x\n' + 'ACGT' * 30000, row.antibiotic, hits=hits, counts=gp.genome_counts('>x\n' + 'ACGT' * 30000))
p = float(served.model.predict(X)[0])
print(f"genome {row.genome_id} + {row.antibiotic}: genes found {list(present)[:8]}{' ...' if len(present) > 8 else ''}")
print(f'website probability {p:.6f} | training run {row.y_score:.6f} | lab result: {"Resistant" if row.y_true else "Susceptible"}')

[Genome] Loaded G_kmer_deploy from genome/.
[Genome] Loaded G_genes_deploy from genome_genes/ (AMRFinderPlus (not needed here)).
served gene model: G_genes_deploy | features: 358


genome 106654.496 + ciprofloxacin: genes found ['blaADC-130']
website probability 0.037286 | training run 0.037286 | lab result: Susceptible


In [9]:
# Species from the 6-mer profile (used to choose AMRFinderPlus's --organism)
ids, counts, length, gc = kmers.load_counts()
i = int(np.flatnonzero(ids == row.genome_id)[0])
print('identified as:', served.species_profiles.identify(counts[i]))

identified as: {'species': np.str_('Acinetobacter nosocomialis'), 'genus': np.str_('Acinetobacter'), 'distance': 0.00017}
